# Phase B Archetype Re-clustering — 2026-04-30

Three tracks: Gower+hierarchical, mixed-type GMM, supervised mood classifier. Decision rules in spec.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from IPython.display import display
import gower
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform

DATA = Path('../../../data/extracted.json')
MOODS = Path('mood_labels.csv')
FIGS = Path('../figures/2026-04-30-phase-b')
FIGS.mkdir(parents=True, exist_ok=True)

CONT_VARS = ['btn_radius','card_radius','heading_weight','body_line_height',
             'heading_letter_spacing','brand_l','brand_c','brand_h','gray_chroma','accent_offset']
ORD_VARS  = ['shadow_intensity','btn_shape']
BOOL_VARS = ['is_fully_pill','dark_mode_present']
ALL_VARS  = CONT_VARS + ORD_VARS + BOOL_VARS

In [2]:
df = pd.DataFrame(json.loads(DATA.read_text()))
moods = pd.read_csv(MOODS)
df = df.merge(moods, on='system', how='left')
print(f'rows: {len(df)}; mood-labeled: {df["mood"].notna().sum()}')
print('\nFailure rate per variable:')
for v in ALL_VARS:
    null = df[v].isna().sum()
    print(f'  {v:24s} {null}/{len(df)} ({null/len(df)*100:.1f}%)')
print('\nMood distribution:')
print(df['mood'].value_counts().to_string())
print('\nis_fully_pill distribution:')
print(df['is_fully_pill'].value_counts(dropna=False).to_string())

rows: 58; mood-labeled: 58

Failure rate per variable:
  btn_radius               2/58 (3.4%)
  card_radius              20/58 (34.5%)
  heading_weight           1/58 (1.7%)
  body_line_height         2/58 (3.4%)
  heading_letter_spacing   7/58 (12.1%)
  brand_l                  1/58 (1.7%)
  brand_c                  1/58 (1.7%)
  brand_h                  1/58 (1.7%)
  gray_chroma              10/58 (17.2%)
  accent_offset            20/58 (34.5%)
  shadow_intensity         17/58 (29.3%)
  btn_shape                1/58 (1.7%)
  is_fully_pill            1/58 (1.7%)
  dark_mode_present        1/58 (1.7%)

Mood distribution:
mood
bold_energetic      17
clean_minimal       13
professional        12
playful_creative     9
warm_friendly        7

is_fully_pill distribution:
is_fully_pill
False    56
True      1
None      1
